# XRZ — Agent-turn Merkle checkpoints

**Synthetic fixture only.** Demonstrates independently reconstructed ordered peaks/root and detection of mutation, reordering, truncation and wrong predecessor root. Does **not** prove actual Copilot execution, independent hardware or signing.

## Context and method

Source: `custody/agent_turn_mmr.py`. Schema: `contracts/AGENT_TURN.schema.json`. Domain-separated SHA-256; UTF-8/NFC integer-only compact canonical JSON (`xrz-json-v1`), not RFC8785. No private lab/patient data. Only local dual-algorithm verification.

In [ ]:
from pathlib import Path
import sys, copy, json
repo = Path.cwd().resolve()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))
from custody.agent_turn_mmr import construct_turn_checkpoint, reference_verify
print("Local reference file:", repo / "custody" / "agent_turn_mmr.py")

## Input observations — synthetic, not actual GitHub Copilot telemetry

In [ ]:
events = []
for i, kind in enumerate(["synthetic_instructions_read","synthetic_code_proposal","synthetic_test_failure"]):
    events.append({"schema":"xrz.agent_turn.v1",
       "event_id":"SYNTHETIC-"+str(i),
       "actor":"SYNTHETIC_COPILOT",
       "turn_id":"synthetic-turn-001","repo":"biobitworks/xrz",
       "task_ref":"synthetic-local-fixture","sequence":i,
       "status":"SYNTHETIC",
       "input_occurrences":["source-occurrence-0"],
       "output_occurrences":["fixture-occurrence-"+str(i)],
       "evidence_refs":["synthetic:"+kind]})
assert len(events)==3
print("Fixture count:",len(events))

## Deterministic construction and dual-codepath verification

In [ ]:
checkpoint=construct_turn_checkpoint(events, parent_checkpoint="ab"*32)
assert reference_verify(events,checkpoint)
print(json.dumps({"root":checkpoint["root"],"peaks":checkpoint["peaks"],"leaf_count":checkpoint["event_count"],"signature_state":checkpoint["signature_state"],"state":checkpoint["state"],"local_verifier":"PASS_SINGLE_HOST"},indent=2))

## Negative controls — deliberately corrupted input

In [ ]:
changed=copy.deepcopy(events)
changed[1]["status"]="OBSERVED"
reordered=copy.deepcopy(events)
reordered[0],reordered[1]=reordered[1],reordered[0]
wrong_parent=dict(checkpoint,parent_checkpoint="cd"*32)
checks={"mutation_rejected":not reference_verify(changed,checkpoint),
"reorder_rejected":not reference_verify(reordered,checkpoint),
"truncation_rejected":not reference_verify(events[:-1],checkpoint),
"parent_substitution_rejected":not reference_verify(events,wrong_parent)}
assert all(checks.values()),checks
print(json.dumps(checks,indent=2))

## Takeaways and NEXT_ACTION

- The only committed fixture here is **synthetic**. The SHA and peaks commit ordered leaf contents and a parent-root reference, not authorship, science or genuine Copilot activity.
- **Reproduction:** from repository root run `PYTHONPATH=. python -m unittest discover -s tests -v` then `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/00_agent_turn_merkle_breakpoints.ipynb` after installing `nbformat nbclient ipykernel`.
- **Known gaps:** two-host verification, authenticated parent history and append-only consistency, actual Copilot task/review receipt, signatures, mobile and XR tests, biosafety approvals.
- **NEXT_ACTION:** request real Copilot PR review and capture externally source-bound event occurrences before promoting L1 to L2 project custody.